In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os

import pyrootutils

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
os.environ.setdefault("TF_XLA_FLAGS", "--tf_xla_auto_jit=0")
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)



In [ ]:
from building.scaling import (
    ScalingRunConfig,
    load_full_arrays,
    run_experiments,
    summarize_results,
    plot_summary,
)

In [ ]:
COLLECTION = "diff_species"
N_SAMPLES = 20
EPOCHS = 30
PATIENCE = 3
BATCH_SIZE = 32
SEED = 42
THRESHOLD = 0.5
BUILD_MODEL = "cnn2d"
INPUT_REPR = "mel"
MODELS_DIR = ROOT / "models" / "scaling_mel"
RESULTS_FILE = MODELS_DIR / "results.jsonl"

In [ ]:
config = ScalingRunConfig(
    collection=COLLECTION,
    build_model=BUILD_MODEL,
    epochs=EPOCHS,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    threshold=THRESHOLD,
    models_dir=MODELS_DIR,
    results_file=RESULTS_FILE,
    input_repr=INPUT_REPR,
)

arrays = load_full_arrays(
    collection=COLLECTION,
    batch_size=BATCH_SIZE,
    seed=SEED,
    input_repr=INPUT_REPR,
)

print(f"Loaded {len(arrays.class_names)} classes:")
print(arrays.class_names)

In [ ]:
baseline_rows = run_experiments(arrays, config, run_baseline=True, run_scaling=False)
print(f"New baseline runs: {len(baseline_rows)}")
if baseline_rows:
    print("Last baseline row:")
    print(baseline_rows[-1])

In [ ]:
import pandas as pd

def print_per_class_baseline_results(arrays, results_file):
    try:
        df = pd.read_json(results_file, lines=True)
    except Exception as e:
        print(f"Could not read results file: {e}")
        return

    baseline_df = df[df["run_type"] == "baseline"].copy()
    class_names = list(arrays.class_names)

    # Calculate the longest class name
    max_cls_len = max((len(str(cls)) for cls in class_names), default=0)
    col_width = max(max_cls_len + 2, 15)

    header = f"{'Target Class':<{col_width}} | {'Precision':>9} | {'Recall':>9} | {'Epochs':>6} | {'Timestamp'}"
    print(header)
    print("-" * len(header))

    for cls in class_names:
        cls_str = f"'{cls}'"
        matches = baseline_df[baseline_df["target_class"] == cls]
        
        row = matches.iloc[-1]
        prec = row.get("precision")
        rec = row.get("recall")
        epochs = row.get("epochs_trained")
        timestamp = row.get("timestamp")
        # Print the aligned single row
        print(f"{cls_str:<{col_width}} | {prec:>9.4f} | {rec:>9.4f} | {epochs:>6.0f} | {timestamp!s}")

print_per_class_baseline_results(arrays, RESULTS_FILE)

In [ ]:
scaling_rows = run_experiments(
    arrays=arrays,
    config=config,
    n_samples=N_SAMPLES,
    k_values=range(2, len(arrays.class_names)),
    run_baseline=False,
    run_scaling=True,
)
print(f"New scaling runs: {len(scaling_rows)}")
if scaling_rows:
    print("Last scaling row:")
    print(scaling_rows[-1])

In [ ]:
baseline_metrics, summary_df = summarize_results(RESULTS_FILE)
print(f"Baseline recall: {baseline_metrics.recall:.4f}")
print(f"Baseline precision: {baseline_metrics.precision:.4f}")
summary_df

In [ ]:
plot_summary(summary_df, baseline=baseline_metrics)